# Auditoría y Consolidación del Padrón de MCPs

**Objetivo:** Construir una base maestra a partir de ocho envíos del padrón de MCPs, identificar incidencias automáticamente (homónimos, cambios de código, cambios de distrito) y generar archivos de revisión.

---
| Etapa | Descripción |
|-------|-------------|
| 1 | Carga de archivos |
| 2 | Revisión estructural |
| 3 | Limpieza y normalización |
| 4 | Construcción de `df_long` (historial) |
| 5 | Auditoría de llaves |
| 6 | Construcción de `df_consolidado` |
| 7 | Clasificación de incidencias |
| 8 | Resultados y exportación |

---
## Etapa 0 — Imports y configuración

In [ ]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

DATABASES_DIR = Path("DATABASES")

print(f"Carpeta de datos: {DATABASES_DIR.resolve()}")
print(f"Archivos encontrados: {len(list(DATABASES_DIR.glob('*.xlsx')))}")

---
## Etapa 1 — Carga de archivos

Se leen los 8 archivos Excel desde la carpeta `DATABASES/` y se almacenan en un diccionario.

In [ ]:
archivos = sorted(DATABASES_DIR.glob("*.xlsx"))

dfs = {}
for archivo in archivos:
    nombre = archivo.stem
    dfs[nombre] = pd.read_excel(archivo)
    print(f"  ✓  {nombre}  →  {dfs[nombre].shape}")

print(f"\nTotal: {len(dfs)} archivos cargados")

---
## Etapa 2 — Revisión estructural

Se verifica que todos los archivos tengan la misma estructura lógica.

In [ ]:
for nombre, df in dfs.items():
    print(f"  {nombre:<40}  filas={df.shape[0]:>5,}  cols={list(df.columns)}")

In [ ]:
print("\nRevisión de COD_MCP_RENIEC:")
for nombre, df in dfs.items():
    col = "COD_MCP_RENIEC"
    if col not in df.columns:
        print(f"  ⚠ {nombre}: columna ausente")
        continue
    s = df[col]
    print(
        f"  {nombre:<40}  dtype={str(s.dtype):<10}  "
        f"nulos={s.isna().sum()}  únicos={s.nunique()}  "
        f"ejemplo={s.dropna().iloc[0] if len(s.dropna()) > 0 else 'N/A'}"
    )

---
## Etapa 3 — Limpieza y normalización

Se normalizan:
- **`COD_MCP_RENIEC`**: de `float` a cadena de 9 dígitos con cero a la izquierda.
- **Variables de texto** (`DEPARTAMENTO`, `PROVINCIA`, `DISTRITO`, `MCP`): pipeline de cuatro pasos.

### Pipeline de normalización de texto

| Paso | Función | Por qué |
|------|---------|---------|
| 1 | `str.strip()` | Eliminar espacios laterales |
| 2 | `unescape_excel_xml()` | openpyxl serializa U+0081 como `_x0081_` en el XML; hay que revertirlo primero |
| 3 | `fix_garbled()` | Algunos archivos tienen `Ñ`, `Á`, `Ú`, etc. codificados como bytes UTF-8 malinterpretados en cp1252 (ej. `Ñ` → `Ã'`). Se aplica **antes** del upper() porque uppercasing cambia los bytes del caracter afectado (ej. `š`→`Š` en cp1252: 0x9A→0x8A, haciendo que `Ú` decodifique como `Ê`) |
| 4 | `str.upper()` + NFC | Normalización final |

> **Resultado:** sin el fix había **29** códigos con múltiples nombres por artefacto de encoding. Después del fix quedan **6** casos genuinos que requieren revisión manual.

In [ ]:
def unescape_excel_xml(s: str) -> str:
    """Convierte escapes _xHHHH_ de openpyxl al carácter Unicode correspondiente."""
    return re.sub(r'_x([0-9A-Fa-f]{4})_', lambda m: chr(int(m.group(1), 16)), s)


def fix_garbled(s: str) -> str:
    """Repara mojibake: bytes UTF-8 malinterpretados como cp1252.

    Estrategia: encode cada caracter de vuelta a bytes (los C1 controls usan su
    valor ordinal directamente; el resto usa cp1252), luego decodifica como UTF-8.
    Si algo falla, devuelve la cadena sin modificar.

    IMPORTANTE: debe aplicarse ANTES de upper() para no alterar los bytes de los
    caracteres afectados.
    """
    try:
        buf = bytearray()
        for c in s:
            o = ord(c)
            if 0x0080 <= o <= 0x009F:
                buf.append(o)  # C1 controls: valor del code point = byte
            else:
                try:
                    buf.extend(c.encode("cp1252"))
                except UnicodeEncodeError:
                    return s  # no encodeable → no es mojibake
        return buf.decode("utf-8")
    except (UnicodeDecodeError, Exception):
        return s


def normalize_text(x) -> str:
    """Pipeline completo: strip → unescape XML → fix mojibake → upper → NFC."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = unescape_excel_xml(s)
    s = fix_garbled(s)        # ANTES del upper()
    s = s.upper()
    s = unicodedata.normalize("NFC", s)
    return s


def normalize_cod(x) -> str:
    """Convierte COD_MCP_RENIEC a cadena de 9 dígitos con cero a la izquierda."""
    if pd.isna(x):
        return pd.NA
    try:
        return str(int(float(x))).zfill(9)
    except (ValueError, TypeError):
        return pd.NA


# Aplicar a todos los DataFrames
for nombre, df in dfs.items():
    dfs[nombre] = df.copy()

    if "COD_MCP_RENIEC" in dfs[nombre].columns:
        dfs[nombre]["COD_MCP_RENIEC"] = dfs[nombre]["COD_MCP_RENIEC"].apply(normalize_cod)

    for col in ["DEPARTAMENTO", "PROVINCIA", "DISTRITO", "MCP"]:
        if col in dfs[nombre].columns:
            dfs[nombre][col] = dfs[nombre][col].apply(normalize_text)

print("✅ Limpieza completada")

In [ ]:
# Verificar distribución de longitudes del código después de la limpieza
for nombre, df in dfs.items():
    lens = df["COD_MCP_RENIEC"].dropna().str.len().value_counts().to_dict()
    print(f"  {nombre:<40}  longitudes={lens}")

---
## Etapa 4 — Construcción de `df_long` (historial)

Se concatenan todos los archivos en un único DataFrame. Cada fila representa **una MCP en un envío determinado**.

Se añaden las columnas:
- `ENVIO`: número de orden del envío (01–08).
- `ARCHIVO`: nombre del archivo fuente.

In [ ]:
lista = []
equivalencia = []

for i, (nombre, df) in enumerate(dfs.items(), start=1):
    envio = f"{i:02d}"
    df = df.copy()

    if "COUNT(1)" in df.columns:
        df = df.rename(columns={"COUNT(1)": "CANTIDAD"})

    df["ENVIO"] = envio
    df["ARCHIVO"] = nombre
    lista.append(df)

    equivalencia.append({"ENVIO": envio, "ARCHIVO": nombre})

df_long = pd.concat(lista, ignore_index=True)
equivalencia = pd.DataFrame(equivalencia)

print(f"df_long: {df_long.shape[0]:,} filas × {df_long.shape[1]} columnas")
print("\nEquivalencia de envíos:")
display(equivalencia)

---
## Etapa 5 — Auditoría de llaves

Antes de consolidar, se evalúan tres posibles fuentes de incidencias:

1. **Homónimos**: mismo nombre de MCP en distintas provincias del mismo departamento.
2. **Múltiples códigos por nombre**: una misma MCP (DEPTO + PROV + nombre) tiene más de un código RENIEC.
3. **Múltiples nombres por código**: un mismo código RENIEC aparece asociado a más de un nombre.

In [ ]:
# ── 5a. Homónimos ────────────────────────────────────────────────────────────
homonimos = (
    df_long
    .groupby(["DEPARTAMENTO", "MCP"])
    .agg(
        n_provincias=("PROVINCIA", "nunique"),
        provincias=("PROVINCIA", lambda x: " | ".join(sorted(set(x))))
    )
    .reset_index()
    .query("n_provincias > 1")
)

print(f"Nombres de MCP que aparecen en más de una provincia del mismo departamento: {len(homonimos)}")
display(homonimos.head(10))

In [ ]:
# ── 5b. Múltiples códigos para el mismo nombre ───────────────────────────────
multi_cod = (
    df_long
    .groupby(["DEPARTAMENTO", "PROVINCIA", "MCP"])
    .agg(
        n_codigos=("COD_MCP_RENIEC", "nunique"),
        codigos=("COD_MCP_RENIEC", lambda x: " | ".join(sorted(set(x.dropna()))))
    )
    .reset_index()
    .query("n_codigos > 1")
)

print(f"MCPs con más de un código RENIEC (posibles cambios): {len(multi_cod)}")
display(multi_cod.head(10))

In [ ]:
# ── 5c. Múltiples nombres para el mismo código ───────────────────────────────
# Nota: tras la corrección de encoding, estos casos deberían reducirse 
# significativamente ya que la mayoría eran artefactos tipográficos (Ã' vs Ñ).
multi_nombre = (
    df_long
    .groupby("COD_MCP_RENIEC")
    .agg(
        n_mcp=("MCP", "nunique"),
        mcps=("MCP", lambda x: " | ".join(sorted(set(x))))
    )
    .reset_index()
    .query("n_mcp > 1")
)

print(f"Códigos RENIEC con más de un nombre de MCP: {len(multi_nombre)}")
display(multi_nombre)

---
## Etapa 6 — Construcción de `df_consolidado`

Se genera una fila única por MCP (identificada por DEPARTAMENTO + PROVINCIA + MCP). Para cada uno de los 8 envíos se incorporan columnas `COD_XX`, `DIST_XX`, `CANT_XX`.

Columnas auxiliares calculadas:
- `COD_ACTUAL` / `DIST_ACTUAL`: último valor conocido (forward-fill).
- `N_ENVIOS`: en cuántos envíos aparece la MCP.
- `CAMBIO_CODIGO`: número de códigos distintos observados.
- `CAMBIO_DISTRITO`: número de distritos distintos observados.

In [ ]:
base = (
    df_long[["DEPARTAMENTO", "PROVINCIA", "MCP"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

df_consolidado = base.copy()
envios = sorted(df_long["ENVIO"].unique())

for envio in envios:
    aux = (
        df_long[df_long["ENVIO"] == envio]
        [["DEPARTAMENTO", "PROVINCIA", "MCP", "COD_MCP_RENIEC", "DISTRITO", "CANTIDAD"]]
        .rename(columns={
            "COD_MCP_RENIEC": f"COD_{envio}",
            "DISTRITO":       f"DIST_{envio}",
            "CANTIDAD":       f"CANT_{envio}",
        })
    )
    df_consolidado = df_consolidado.merge(
        aux, how="left", on=["DEPARTAMENTO", "PROVINCIA", "MCP"]
    )

cod_cols  = sorted([c for c in df_consolidado.columns if c.startswith("COD_")])
dist_cols = sorted([c for c in df_consolidado.columns if c.startswith("DIST_")])

df_consolidado["COD_ACTUAL"]     = df_consolidado[cod_cols].ffill(axis=1).iloc[:, -1]
df_consolidado["DIST_ACTUAL"]    = df_consolidado[dist_cols].ffill(axis=1).iloc[:, -1]
df_consolidado["N_ENVIOS"]        = df_consolidado[cod_cols].notna().sum(axis=1)
df_consolidado["CAMBIO_CODIGO"]   = df_consolidado[cod_cols].nunique(axis=1, dropna=True)
df_consolidado["CAMBIO_DISTRITO"] = df_consolidado[dist_cols].nunique(axis=1, dropna=True)

print(f"df_consolidado: {df_consolidado.shape[0]:,} filas × {df_consolidado.shape[1]} columnas")
display(df_consolidado.head(3))

In [ ]:
# Agregar homónimos
hom_merge = (
    df_long
    .groupby(["DEPARTAMENTO", "MCP"])
    .agg(
        N_PROVINCIAS=("PROVINCIA", "nunique"),
        PROVINCIAS=("PROVINCIA", lambda x: " | ".join(sorted(set(x))))
    )
    .reset_index()
)
df_consolidado = df_consolidado.merge(hom_merge, on=["DEPARTAMENTO", "MCP"], how="left")

# Agregar múltiples códigos
cod_merge = (
    df_long
    .groupby(["DEPARTAMENTO", "PROVINCIA", "MCP"])
    .agg(
        N_CODIGOS=("COD_MCP_RENIEC", "nunique"),
        CODIGOS=("COD_MCP_RENIEC", lambda x: " | ".join(sorted(set(x.dropna()))))
    )
    .reset_index()
)
df_consolidado = df_consolidado.merge(cod_merge, on=["DEPARTAMENTO", "PROVINCIA", "MCP"], how="left")

print("✅ Columnas de auditoría agregadas")

---
## Etapa 7 — Clasificación de incidencias

Se genera la columna `OBSERVACIONES` con todas las incidencias que aplican a cada MCP (una MCP puede tener varias simultáneamente):

| Etiqueta | Condición |
|---|---|
| `OK` | Sin incidencias |
| `HOMONIMO` | El mismo nombre aparece en >1 provincia del mismo departamento |
| `CAMBIO_CODIGO` | La MCP tuvo >1 código RENIEC entre envíos |
| `CAMBIO_DISTRITO` | La MCP apareció en >1 distrito entre envíos |

In [ ]:
def clasificar_incidencias(row) -> str:
    obs = []
    if row["N_PROVINCIAS"] > 1:
        obs.append("HOMONIMO")
    if row["N_CODIGOS"] > 1:
        obs.append("CAMBIO_CODIGO")
    if row["CAMBIO_DISTRITO"] > 1:
        obs.append("CAMBIO_DISTRITO")
    return " | ".join(obs) if obs else "OK"


df_consolidado["OBSERVACIONES"]     = df_consolidado.apply(clasificar_incidencias, axis=1)
df_consolidado["REQUIERE_REVISION"] = df_consolidado["OBSERVACIONES"] != "OK"

print("Distribución de observaciones:")
display(df_consolidado["OBSERVACIONES"].value_counts().to_frame())

total_revision = df_consolidado["REQUIERE_REVISION"].sum()
total          = len(df_consolidado)
print(f"\nMCPs que requieren revisión: {total_revision:,} de {total:,} ({total_revision/total*100:.1f}%)")

In [ ]:
# df_revision: subconjunto de df_consolidado con solo los casos problemáticos
df_revision = (
    df_consolidado
    .query("REQUIERE_REVISION")
    .sort_values(["OBSERVACIONES", "DEPARTAMENTO", "PROVINCIA", "MCP"])
    .reset_index(drop=True)
)

print(f"df_revision: {len(df_revision):,} filas")
display(df_revision[["DEPARTAMENTO","PROVINCIA","MCP","COD_ACTUAL","DIST_ACTUAL",
                      "N_ENVIOS","N_CODIGOS","CAMBIO_DISTRITO","OBSERVACIONES"]].head(10))

---
## Etapa 8 — Resultados y exportación

Se exportan tres archivos:
- `df_consolidado.xlsx`: base maestra completa (una fila por MCP).
- `df_revision.xlsx`: solo los casos que requieren revisión manual.
- `df_long.xlsx`: historial completo (una fila por MCP × envío).

In [ ]:
# Columnas para la exportación de df_consolidado
cols_export = (
    ["DEPARTAMENTO", "PROVINCIA", "MCP", "COD_ACTUAL", "DIST_ACTUAL",
     "N_ENVIOS", "N_CODIGOS", "CODIGOS", "CAMBIO_DISTRITO",
     "N_PROVINCIAS", "PROVINCIAS", "OBSERVACIONES", "REQUIERE_REVISION"]
    + [c for c in df_consolidado.columns if c.startswith("COD_") and c != "COD_ACTUAL"]
    + [c for c in df_consolidado.columns if c.startswith("DIST_") and c != "DIST_ACTUAL"]
    + [c for c in df_consolidado.columns if c.startswith("CANT_")]
)
cols_export = [c for c in cols_export if c in df_consolidado.columns]  # por seguridad

df_consolidado[cols_export].to_excel("df_consolidado.xlsx", index=False)
df_revision[[c for c in cols_export if c in df_revision.columns]].to_excel("df_revision.xlsx", index=False)
df_long.to_excel("df_long.xlsx", index=False)

print("✅ Exportación completada:")
print("   • df_consolidado.xlsx")
print("   • df_revision.xlsx")
print("   • df_long.xlsx")

---
## Resumen final

| Resultado | Valor |
|---|---|
| Total de envíos procesados | 8 |
| Total de registros (`df_long`) | — |
| Total de MCPs únicas (`df_consolidado`) | — |
| MCPs sin incidencias | — |
| MCPs que requieren revisión | — |

### Trabajo pendiente
1. Resolver manualmente `MAURE KALLACHIRI` vs `MAURE KALLAPUMA` (código `220201001`).
2. Validar si los **cambios de código** corresponden a actualización oficial RENIEC o error de captura.
3. Validar si los **cambios de distrito** son correcciones reales o errores.
4. Construir un identificador permanente de MCP una vez resueltas las incidencias.